# Tutorial 2: The MELV Master Equation — Peer Review Reproducibility Notebook

**Author:** Laurence W. Evans | ORCID: 0009-0001-0963-1840  
**DOI (data):** [10.5281/zenodo.19422174](https://doi.org/10.5281/zenodo.19422174)  
**Preprint:** [10.5281/zenodo.19029077](https://doi.org/10.5281/zenodo.19029077)

---

This notebook reproduces **all six prediction tests** reported in Evans (2026) using the ABM V2.1 validation dataset (405 simulation runs). No installation is required — all data is included in this repository and all analysis runs here in the browser.

## What This Notebook Covers

| Part | Content |
|------|---------|
| 1 | The MELV master equation — theory and parameters |
| 2 | Dataset overview — 405 runs, parameter space |
| 3 | Prediction 1 — Bimodal outcome distribution (Hartigan dip test) |
| 4 | Prediction 2 — Threshold universality (ANOVA across ε) |
| 5 | Prediction 3 — Crossing time vs ε (null result, explained) |
| 6 | Prediction 4 — Negative correlation: mean i vs cooperation |
| 7 | Prediction 5 — φ×β predicts cooperative crossing |
| 8 | Prediction 6 — ESS attractor stability (invasion recovery) |
| 9 | Run a single simulation yourself |

---

> **Relationship to Tutorial 1:** Tutorial 1 (in the melv-core repository) introduces a pedagogical simplification — the ratio model i = ρ/δ. This tutorial implements the **full 2026 master equation** with temporal dynamics. The two models are related in spirit but mechanistically distinct.

In [ ]:
# Install dependencies if running on Binder or fresh environment
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import diptest
except ImportError:
    install('diptest')

try:
    import mesa
except ImportError:
    install('mesa==3.5.1')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import diptest
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✓ All dependencies loaded')
print(f'  numpy   {np.__version__}')
print(f'  pandas  {pd.__version__}')
print(f'  diptest {diptest.__version__}')

## Part 1: The MELV Master Equation

The Modified Energetic Lotka-Volterra (MELV) framework extends classical Lotka-Volterra dynamics with a dimensionless interaction coefficient i₁₂(t) that governs the quality and direction of inter-species energy exchange.

### Master equation

```
i₁₂(t) = i₁₂⁰ × (1 − ε × φ(t) × β(t))
```

| Symbol | Biological meaning | ABM implementation |
|--------|--------------------|--------------------|
| i₁₂(t) | Compatibility factor at time t | `agent.i_factor` |
| i₁₂⁰   | Initial compatibility | Drawn from prior: `1 − 0.3×tanh(R×I_freq − 1)` |
| ε       | Adaptation acceleration (evolutionary rate) | Parameter: {0.05, 0.10, 0.20} |
| φ(t)    | Perpetuity — long-run relational stability | `agent.phi` (co-evolves) |
| β(t)    | Cooperative neighbourhood density | Fraction of cooperative Moore neighbours |
| φ×β     | Joint cooperation driver | `agent.phi_beta` |

### Prediction

The equation is dimensionless — i₁₂(t) functions analogously to a refractive index. Full cooperative convergence (Cooperation Index CI → 1.0) occurs asymptotically as i → 1 from above, confirmed empirically at i = 0.9995 ± 0.029.

### The sigmoid Allee effect (V2.1)

Cooperator efficiency uses a logistic function of local β:

```python
efficiency_coop = 1.0 + (1 − i_factor) × sigmoid(β, k=10, τ=0.5) × φ
```

This is mathematically equivalent to bacterial quorum sensing — cooperation is suppressed below the quorum threshold τ = 0.5, regardless of the agent's i-factor. Below quorum, isolated cooperators cannot sustain mutualism (Allee, 1931; Nadell et al., 2016).

In [ ]:
# Visualise the master equation and sigmoid Allee effect
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Panel A: Master equation surface ---
ax = axes[0]
phi_vals = np.linspace(0, 1, 100)
beta_vals = np.linspace(0, 1, 100)
PHI, BETA = np.meshgrid(phi_vals, beta_vals)
for eps, col, lbl in [(0.05, '#2E86AB', 'ε = 0.05'), 
                       (0.10, '#A23B72', 'ε = 0.10'),
                       (0.20, '#F18F01', 'ε = 0.20')]:
    # i_predicted along the phi=beta diagonal
    diag = np.linspace(0, 1, 200)
    i_pred = 1.0 - eps * diag * diag
    ax.plot(diag, i_pred, color=col, lw=2, label=lbl)
ax.axhline(1.0, color='black', lw=1, ls='--', alpha=0.5)
ax.fill_between([0,1], [0,0], [1,1], alpha=0.08, color='green', label='Cooperation zone (i<1)')
ax.fill_between([0,1], [1,1], [1.5,1.5], alpha=0.08, color='red', label='Competition zone (i>1)')
ax.set_xlabel('φ×β (joint cooperation driver)', fontsize=11)
ax.set_ylabel('i₁₂(t) / i₁₂⁰  [normalised]', fontsize=11)
ax.set_title('Master equation: i predicted\nalong φ=β diagonal', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0.5, 1.3)
ax.set_xlim(0, 1)

# --- Panel B: Sigmoid Allee (quorum sensing) ---
ax = axes[1]
beta_range = np.linspace(0, 1, 300)
for k, col, lbl in [(10, '#2E86AB', 'k=10 (V2.1, τ=0.5)'),
                     (3,  '#A23B72', 'k=3 (shallower)'),
                     (50, '#F18F01', 'k=50 (steeper)')]:
    sig = 1.0 / (1.0 + np.exp(-k * (beta_range - 0.5)))
    ax.plot(beta_range, sig, color=col, lw=2, label=lbl)
ax.axvline(0.5, color='black', lw=1, ls='--', alpha=0.6, label='Quorum threshold τ=0.5')
ax.fill_betweenx([0,1], 0, 0.5, alpha=0.08, color='red', label='Sub-quorum (cooperation suppressed)')
ax.fill_betweenx([0,1], 0.5, 1.0, alpha=0.08, color='green', label='Above quorum (cooperation active)')
ax.set_xlabel('Local β (cooperative neighbour density)', fontsize=11)
ax.set_ylabel('sigmoid(β, k, τ=0.5)', fontsize=11)
ax.set_title('Sigmoid Allee / Quorum-sensing gate\n(fixed biological constants, not free parameters)', 
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.suptitle('MELV V2.1 — Core Mathematical Structure', 
             fontsize=13, fontweight='bold', y=1.02)
plt.show()
print('Figure: Master equation (left) and sigmoid Allee/quorum gate (right)')

## Part 2: Dataset Overview

405 simulation runs across a 3×3×3×3 parameter grid, 5 replicates per condition, 2000 steps per run.

In [ ]:
import os

# Load validation summary (405 runs)
# Try local path first, then data/ subdirectory
for path in ['validation_summary_v2_1_2000.csv', 
             'data/validation_summary_v2_1_2000.csv']:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Loaded from: {path}')
        break

print(f'\nDataset: {len(df)} runs × {len(df.columns)} columns')
print(f'\nParameter space:')
print(f'  R (resource abundance):   {sorted(df.R.unique())}')
print(f'  I_freq (interaction freq): {sorted(df.I_freq.unique())}')
print(f'  K (carrying capacity):     {sorted(df.K.unique())}')
print(f'  ε (adaptation rate):       {sorted(df.epsilon.unique())}')
print(f'  Replicates per condition:  5')
print(f'\nOutcome distribution:')
for outcome, count in df.outcome.value_counts().items():
    pct = count / len(df) * 100
    print(f'  {outcome:6s}: {count:3d} runs ({pct:.0f}%)')
print(f'\nKey metrics range:')
print(f'  final_mean_i:          {df.final_mean_i.min():.3f} – {df.final_mean_i.max():.3f}')
print(f'  final_cooperation_level: {df.final_cooperation_level.min():.3f} – {df.final_cooperation_level.max():.3f}')
print(f'  final_mean_phi_beta:   {df.final_mean_phi_beta.min():.3f} – {df.final_mean_phi_beta.max():.3f}')

## Part 3: Prediction 1 — Bimodal Outcome Distribution

**Prediction:** If the MELV master equation correctly describes a bifurcation near i → 1, the distribution of final mean i-factors across runs should be **bimodal** — one peak in the competitive basin (i > 1) and one in the cooperative basin (i < 1), with relatively few runs near the boundary.

**Test:** Hartigan's dip test for unimodality. Rejection (p < 0.05) confirms bimodality.

**Reported result:** dip = 0.047, p ≈ 0

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Hartigan dip test
dip_stat, pval = diptest.diptest(df['final_mean_i'].values)
print(f'Hartigan dip test:')
print(f'  dip statistic = {dip_stat:.4f}')
print(f'  p-value       = {pval:.2e}')
print(f'  Bimodality confirmed: {"YES" if pval < 0.05 else "NO"}')

# --- Panel A: Distribution of final mean i ---
ax = axes[0]
coop_mask = df.outcome == 'COOP'
comp_mask = df.outcome == 'COMP'
thr_mask  = df.outcome == 'THRESH'

ax.hist(df.loc[comp_mask, 'final_mean_i'], bins=30, color='#E63946', 
        alpha=0.75, label=f'COMP (n={comp_mask.sum()})', density=True)
ax.hist(df.loc[coop_mask, 'final_mean_i'], bins=30, color='#2A9D8F', 
        alpha=0.75, label=f'COOP (n={coop_mask.sum()})', density=True)
ax.hist(df.loc[thr_mask, 'final_mean_i'],  bins=30, color='#E9C46A', 
        alpha=0.75, label=f'THRESH (n={thr_mask.sum()})', density=True)
ax.axvline(0.9, color='black', lw=1.5, ls='--', alpha=0.7, label='Basin boundaries')
ax.axvline(1.1, color='black', lw=1.5, ls='--', alpha=0.7)
ax.set_xlabel('Final mean i-factor', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title(f'Bimodal distribution of outcomes\nHartigan dip = {dip_stat:.3f}, p ≈ {pval:.1e}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# --- Panel B: Outcome breakdown by R ---
ax = axes[1]
outcome_by_R = df.groupby(['R', 'outcome']).size().unstack(fill_value=0)
outcome_by_R_pct = outcome_by_R.div(outcome_by_R.sum(axis=1), axis=0) * 100
outcome_by_R_pct[['COOP','THRESH','COMP']].plot(
    kind='bar', ax=ax, 
    color=['#2A9D8F', '#E9C46A', '#E63946'],
    alpha=0.85, width=0.7, edgecolor='white')
ax.set_xlabel('Resource abundance (R)', fontsize=11)
ax.set_ylabel('% of runs', fontsize=11)
ax.set_title('Outcome by resource level\n(cooperation requires high R)', 
             fontsize=11, fontweight='bold')
ax.set_xticklabels([f'R={r}' for r in sorted(df.R.unique())], rotation=0)
ax.legend(fontsize=9)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## Part 4: Prediction 2 — Threshold Universality

**Prediction:** The bifurcation threshold should be **universal** across different evolutionary rates ε. If i* depends on ε, then the threshold is a model artefact rather than an intrinsic property of the master equation.

**Test:** One-way ANOVA of final mean i across ε strata, restricted to THRESH runs (boundary-dwellers). Non-significant result (p > 0.05) confirms universality.

**Reported result:** F = 1.91, p = 0.15

In [ ]:
# ANOVA of final_mean_i across epsilon values (all runs)
groups = [grp['final_mean_i'].values 
          for _, grp in df.groupby('epsilon')]
F_stat, p_anova = stats.f_oneway(*groups)

print('One-way ANOVA: final mean i across ε strata')
print(f'  F = {F_stat:.3f}')
print(f'  p = {p_anova:.4f}')
print(f'  Threshold universal (p > 0.05): {"YES" if p_anova > 0.05 else "NO"}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Panel A: Boxplot by epsilon ---
ax = axes[0]
colors = ['#2E86AB', '#A23B72', '#F18F01']
eps_vals = sorted(df.epsilon.unique())
data_by_eps = [df[df.epsilon == e]['final_mean_i'].values for e in eps_vals]
bp = ax.boxplot(data_by_eps, patch_artist=True, notch=False,
                medianprops=dict(color='black', lw=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(0.9, color='#2A9D8F', lw=1.5, ls='--', alpha=0.7, label='COOP boundary')
ax.axhline(1.1, color='#E63946', lw=1.5, ls='--', alpha=0.7, label='COMP boundary')
ax.set_xticks(range(1, len(eps_vals)+1))
ax.set_xticklabels([f'ε = {e}' for e in eps_vals])
ax.set_ylabel('Final mean i-factor', fontsize=11)
ax.set_title(f'Threshold universality across ε\nANOVA: F={F_stat:.2f}, p={p_anova:.3f} (NS)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# --- Panel B: Mean cooperation level by epsilon ---
ax = axes[1]
mean_coop = df.groupby('epsilon')['final_cooperation_level'].agg(['mean','sem']).reset_index()
ax.bar(range(len(eps_vals)), mean_coop['mean'], 
       yerr=mean_coop['sem'], capsize=6,
       color=colors, alpha=0.8, edgecolor='black', lw=1.2)
ax.set_xticks(range(len(eps_vals)))
ax.set_xticklabels([f'ε = {e}' for e in eps_vals])
ax.set_ylabel('Mean cooperation level (± SEM)', fontsize=11)
ax.set_title('Cooperation level by ε\n(ε does not determine cooperation outcome)',
             fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Part 5: Prediction 3 — Crossing Time vs ε (Null Result)

**Prediction:** Higher evolutionary rate ε should produce faster crossing to the cooperative basin (shorter crossing time).

**Test:** Linear regression of crossing_step on ε, restricted to COOP runs.

**Reported result:** NULL — resource availability (R) dominates crossing timing, not ε.

**Interpretation:** Consistent with major evolutionary transitions literature (Michod, 1999; Pennisi, 2009) — cooperation transitions are triggered by ecological opportunity, not elevated mutation rate. ε controls drift magnitude but does not override ecological structure.

In [ ]:
coop_df = df[df.outcome == 'COOP'].dropna(subset=['crossing_step'])

if len(coop_df) > 3:
    slope, intercept, r, p_reg, se = stats.linregress(
        coop_df['epsilon'], coop_df['crossing_step'])
    print(f'Linear regression: crossing_step ~ ε (COOP runs only, n={len(coop_df)})')
    print(f'  slope = {slope:.1f}')
    print(f'  r     = {r:.4f}')
    print(f'  p     = {p_reg:.4f}')
    print(f'  Prediction confirmed (p<0.05): {"YES" if p_reg < 0.05 else "NO — null result"}')
else:
    print(f'Only {len(coop_df)} COOP runs with recorded crossing steps.')
    print('Null result: insufficient COOP runs with crossing data for regression.')
    print('Most cooperative runs cross very early (step ~10), giving limited variance.')
    r, p_reg = float('nan'), float('nan')
    slope = float('nan')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Panel A: crossing time vs epsilon ---
ax = axes[0]
for e, col in zip(sorted(df.epsilon.unique()), ['#2E86AB','#A23B72','#F18F01']):
    sub = coop_df[coop_df.epsilon == e]
    ax.scatter(sub['epsilon'] + np.random.normal(0, 0.003, len(sub)), 
               sub['crossing_step'], color=col, alpha=0.6, s=40,
               label=f'ε={e} (n={len(sub)})')
if not np.isnan(slope):
    x_line = np.array(sorted(df.epsilon.unique()))
    ax.plot(x_line, intercept + slope*x_line, 'k--', lw=1.5, 
            label=f'Regression (r={r:.3f}, p={p_reg:.3f})')
ax.set_xlabel('ε (adaptation rate)', fontsize=11)
ax.set_ylabel('Crossing step (COOP runs)', fontsize=11)
ax.set_title('Prediction 3: Crossing time vs ε\nNULL result — ε does not predict crossing timing',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# --- Panel B: crossing time vs R (the actual driver) ---
ax = axes[1]
for R_val, col in zip(sorted(df.R.unique()), ['#E63946','#E9C46A','#2A9D8F']):
    sub = coop_df[coop_df.R == R_val]
    ax.scatter(sub['R'] + np.random.normal(0, 0.04, len(sub)),
               sub['crossing_step'], color=col, alpha=0.7, s=50,
               label=f'R={R_val} (n={len(sub)})')
ax.set_xlabel('R (resource abundance)', fontsize=11)
ax.set_ylabel('Crossing step (COOP runs)', fontsize=11)
ax.set_title('Resource abundance is the actual driver\n(ecological opportunity, not evolutionary rate)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 6: Prediction 4 — Negative Correlation: Mean i vs Cooperation

**Prediction:** Final mean i-factor should be strongly **negatively correlated** with cooperation level. Lower i → more cooperation.

**Test:** Pearson r between `final_mean_i` and `final_cooperation_level`. Criterion: r < −0.90.

**Reported result:** r = −0.866 — strong, criterion slightly missed due to asymmetric distribution (79% COMP runs cluster at high i with low cooperation variance).

In [ ]:
r_pearson, p_corr = stats.pearsonr(df['final_mean_i'], df['final_cooperation_level'])
print(f'Pearson correlation: final_mean_i vs final_cooperation_level')
print(f'  r = {r_pearson:.4f}')
print(f'  p = {p_corr:.2e}')
print(f'  Strong negative (r < -0.90): {"YES" if r_pearson < -0.90 else "NO — r = " + str(round(r_pearson,3))}')
print(f'  Criterion slightly missed due to asymmetric distribution (79% COMP)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Panel A: Scatter with regression ---
ax = axes[0]
colour_map = {'COOP': '#2A9D8F', 'COMP': '#E63946', 'THRESH': '#E9C46A'}
for outcome in ['COMP', 'THRESH', 'COOP']:
    mask = df.outcome == outcome
    ax.scatter(df.loc[mask, 'final_mean_i'],
               df.loc[mask, 'final_cooperation_level'],
               c=colour_map[outcome], alpha=0.5, s=20, label=outcome)
x_fit = np.linspace(df.final_mean_i.min(), df.final_mean_i.max(), 100)
slope_c, intercept_c, _, _, _ = stats.linregress(
    df['final_mean_i'], df['final_cooperation_level'])
ax.plot(x_fit, intercept_c + slope_c * x_fit, 'k--', lw=1.5,
        label=f'r = {r_pearson:.3f} (p < 10⁻¹⁰⁰)')
ax.set_xlabel('Final mean i-factor', fontsize=11)
ax.set_ylabel('Final cooperation level', fontsize=11)
ax.set_title(f'Prediction 4: Negative correlation\nr = {r_pearson:.3f}, p = {p_corr:.1e}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# --- Panel B: Mean cooperation by outcome ---
ax = axes[1]
means = df.groupby('outcome')['final_cooperation_level'].agg(['mean','std','count'])
for i, (outcome, row) in enumerate(means.iterrows()):
    ax.bar(i, row['mean'], 
           yerr=row['std']/np.sqrt(row['count']),
           color=colour_map[outcome], alpha=0.8,
           edgecolor='black', lw=1.2, capsize=6,
           label=f'{outcome} (n={int(row["count"])})')
ax.set_xticks(range(len(means)))
ax.set_xticklabels(means.index)
ax.set_ylabel('Mean cooperation level (± SEM)', fontsize=11)
ax.set_title('Cooperation level by basin\n(COOP basin = fully cooperative)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 7: Prediction 5 — φ×β Predicts Cooperative Crossing

**Prediction:** The φ×β product (perpetuity × neighbourhood density) should be the primary predictor of whether a run crosses to the cooperative basin. This validates the multiplicative structure of the master equation — φ and β are synergistic, not additive.

**Test:** Logistic regression classifying COOP vs COMP/THRESH using `final_mean_phi_beta`. Sensitivity and specificity both > 0.75.

**Reported result:** sensitivity = 1.0, specificity = 0.997

In [ ]:
from scipy.special import expit

# Binary classification: COOP=1, other=0
df['is_coop'] = (df.outcome == 'COOP').astype(int)

# Logistic regression via scipy
X = df['final_mean_phi_beta'].values
y = df['is_coop'].values

# Find optimal threshold via logistic fit
from scipy.optimize import minimize_scalar

def neg_log_lik(threshold):
    pred = (X >= threshold).astype(int)
    # Penalise misclassification
    return -( np.sum(y * np.log(pred.clip(1e-9))) + 
               np.sum((1-y) * np.log((1-pred).clip(1e-9))) )

# Simple threshold scan
best_thresh, best_sens, best_spec, best_f1 = 0, 0, 0, 0
for thresh in np.linspace(0.01, 0.99, 500):
    pred = (X >= thresh).astype(int)
    tp = np.sum((pred == 1) & (y == 1))
    tn = np.sum((pred == 0) & (y == 0))
    fp = np.sum((pred == 1) & (y == 0))
    fn = np.sum((pred == 0) & (y == 1))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1   = 2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) > 0 else 0
    if f1 > best_f1:
        best_thresh, best_sens, best_spec, best_f1 = thresh, sens, spec, f1

print(f'φ×β threshold classification (optimal threshold = {best_thresh:.3f})')
print(f'  Sensitivity (COOP recall):  {best_sens:.4f}')
print(f'  Specificity (COMP/THRESH):  {best_spec:.4f}')
print(f'  Both > 0.75: {"YES" if best_sens > 0.75 and best_spec > 0.75 else "NO"}')
print(f'  F1 score:  {best_f1:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Panel A: φ×β distribution by outcome ---
ax = axes[0]
for outcome, col in colour_map.items():
    mask = df.outcome == outcome
    ax.hist(df.loc[mask, 'final_mean_phi_beta'], bins=25,
            color=col, alpha=0.65, density=True, label=outcome)
ax.axvline(best_thresh, color='black', lw=2, ls='--',
           label=f'Optimal threshold = {best_thresh:.2f}')
ax.set_xlabel('Final mean φ×β', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title(f'Prediction 5: φ×β predicts COOP crossing\nsens={best_sens:.3f}, spec={best_spec:.3f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# --- Panel B: φ×β vs cooperation level ---
ax = axes[1]
for outcome in ['COMP','THRESH','COOP']:
    mask = df.outcome == outcome
    ax.scatter(df.loc[mask,'final_mean_phi_beta'],
               df.loc[mask,'final_cooperation_level'],
               c=colour_map[outcome], alpha=0.5, s=20, label=outcome)
ax.axvline(best_thresh, color='black', lw=1.5, ls='--', alpha=0.7,
           label=f'Classification threshold = {best_thresh:.2f}')
ax.set_xlabel('Final mean φ×β', fontsize=11)
ax.set_ylabel('Final cooperation level', fontsize=11)
ax.set_title('Synergistic effect of φ×β\n(multiplicative structure of master equation validated)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Part 8: Prediction 6 — ESS Attractor Stability (Invasion Recovery)

**Prediction:** Both the cooperative and competitive attractors should be **evolutionarily stable** — resistant to invasion by a disruptive mutant agent. A population pushed out of its attractor basin should recover.

**Test:** 34 invasion tests. At peak stability, a 25% mutant population is introduced (Δi ≈ 0.57 displacement). Recovery assessed over 500 post-invasion steps. Criterion: > 85% recovery.

**Reported result:** 34/34 (100%) recovery — both COOP and COMP basins.

In [ ]:
# Load perturbation results
for path in ['perturbation_results.csv', 'data/perturbation_results.csv']:
    if os.path.exists(path):
        pert = pd.read_csv(path)
        print(f'Loaded from: {path}')
        break

print(f'\nInvasion test summary ({len(pert)} tests):')
print(f'  Total tests:       {len(pert)}')
print(f'  Recovered:         {pert.recovered.sum()} ({pert.recovered.mean()*100:.0f}%)')
print(f'  COOP tests:        {(pert.expected=="COOP").sum()}')
print(f'  COMP tests:        {(pert.expected=="COMP").sum()}')
print(f'  COOP recovery:     {pert[pert.expected=="COOP"].recovered.mean()*100:.0f}%')
print(f'  COMP recovery:     {pert[pert.expected=="COMP"].recovered.mean()*100:.0f}%')
print(f'  Mean displacement: Δi = {(pert.post_invasion_mean_i - pert.pre_mean_i).abs().mean():.3f}')
print(f'\n  ESS criterion (>85%): {"PASS" if pert.recovered.mean() > 0.85 else "FAIL"}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Panel A: Pre vs post-invasion vs final i ---
ax = axes[0]
for basin, col, marker in [('COOP','#2A9D8F','o'), ('COMP','#E63946','s')]:
    mask = pert.expected == basin
    sub = pert[mask]
    x = np.arange(len(sub))
    ax.scatter(x, sub.pre_mean_i, color=col, marker=marker, s=60, 
               alpha=0.9, label=f'{basin} pre-invasion')
    ax.scatter(x, sub.post_invasion_mean_i, color=col, marker='x', s=80,
               alpha=0.7, label=f'{basin} post-invasion')
    ax.scatter(x, sub.final_mean_i, color=col, marker=marker, s=60,
               alpha=0.4, edgecolors='black', lw=1)
ax.axhline(0.9, color='#2A9D8F', lw=1.2, ls='--', alpha=0.6, label='Basin boundaries')
ax.axhline(1.1, color='#E63946', lw=1.2, ls='--', alpha=0.6)
ax.set_xlabel('Test index', fontsize=11)
ax.set_ylabel('Mean i-factor', fontsize=11)
ax.set_title(f'Prediction 6: ESS invasion recovery\n34/34 (100%) recovered to original basin',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8, ncol=2)

# --- Panel B: Recovery summary ---
ax = axes[1]
categories = ['COOP\nbasin', 'COMP\nbasin', 'Overall']
values = [
    pert[pert.expected=='COOP'].recovered.mean() * 100,
    pert[pert.expected=='COMP'].recovered.mean() * 100,
    pert.recovered.mean() * 100
]
colors_bar = ['#2A9D8F', '#E63946', '#264653']
bars = ax.bar(categories, values, color=colors_bar, alpha=0.85,
              edgecolor='black', lw=1.2)
ax.axhline(85, color='black', lw=1.5, ls='--', label='ESS criterion (85%)')
ax.axhline(100, color='gray', lw=0.8, ls=':', alpha=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1,
            f'{val:.0f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('Recovery rate (%)', fontsize=11)
ax.set_ylim(0, 115)
ax.set_title('ESS attractor stability\nBoth basins are evolutionarily stable',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## Part 9: Run a Single Simulation Yourself

This cell runs a live MELV ABM simulation using the V2.1 model code. You can modify the parameters and observe how the outcome changes.

**Note:** Requires `mesa==3.5.1`. If running on Binder this is already installed. Typical runtime: ~10 seconds.

In [ ]:
import sys, os
# Add parent dir to path so melv_abm_v2_1 is importable
for p in ['.', '..']:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from melv_abm_v2_1 import MELVModel
    MODEL_AVAILABLE = True
except ImportError as e:
    print(f'Note: MELVModel not importable ({e})')
    print('This is expected if running from the tutorials/ subdirectory.')
    print('The simulation visualisation below uses pre-recorded data instead.')
    MODEL_AVAILABLE = False

if MODEL_AVAILABLE:
    # ── MODIFY THESE PARAMETERS ──────────────────────────────────────────
    R       = 3.0   # Resource abundance: {0.3, 1.0, 3.0}
    I_freq  = 0.8   # Interaction frequency: {0.2, 0.5, 0.8}
    K       = 5     # Carrying capacity: {3, 5, 8}
    epsilon = 0.10  # Adaptation rate: {0.05, 0.10, 0.20}
    N_STEPS = 500   # Steps to run (2000 in full validation)
    SEED    = 42
    # ────────────────────────────────────────────────────────────────────

    print(f'Running MELV ABM V2.1...')
    print(f'  R={R}, I_freq={I_freq}, K={K}, ε={epsilon}, steps={N_STEPS}')
    model = MELVModel(R=R, I_freq=I_freq, K=K, epsilon=epsilon, seed=SEED)
    history = []
    for step in range(N_STEPS):
        model.step()
        data = model.datacollector.get_model_vars_dataframe()
        if step % 50 == 0:
            row = data.iloc[-1]
            history.append({'step': step, 
                            'mean_i': row.get('mean_i', float('nan')),
                            'cooperation': row.get('cooperation_level', float('nan'))})

    final_i = history[-1]['mean_i']
    if final_i < 0.9:
        outcome = 'COOP'
        col = '#2A9D8F'
    elif final_i > 1.1:
        outcome = 'COMP'
        col = '#E63946'
    else:
        outcome = 'THRESH'
        col = '#E9C46A'

    print(f'\nResult:')
    print(f'  Final mean i-factor: {final_i:.4f}')
    print(f'  Outcome: {outcome}')

    hist_df = pd.DataFrame(history)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(hist_df.step, hist_df.mean_i, color=col, lw=2, label='mean i-factor')
    ax.plot(hist_df.step, hist_df.cooperation, color='gray', lw=1.5, 
            ls='--', label='cooperation level')
    ax.axhline(0.9, color='#2A9D8F', lw=1, ls=':', alpha=0.7, label='COOP boundary')
    ax.axhline(1.1, color='#E63946', lw=1, ls=':', alpha=0.7, label='COMP boundary')
    ax.axhline(1.0, color='black', lw=0.8, ls='--', alpha=0.4, label='i = 1.0 (threshold)')
    ax.set_xlabel('Simulation step', fontsize=11)
    ax.set_ylabel('i-factor / cooperation level', fontsize=11)
    ax.set_title(f'Live simulation: R={R}, I_freq={I_freq}, K={K}, ε={epsilon}\n'
                 f'Outcome: {outcome} (final mean i = {final_i:.3f})',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    # Fallback: show a sample trajectory from the recorded data
    print('\nShowing a sample COOP trajectory from recorded validation data...')
    for traj_path in ['validation_data_v2_1_2000.csv', 
                      'data/validation_data_v2_1_2000.csv']:
        if os.path.exists(traj_path):
            traj = pd.read_csv(traj_path, nrows=5000)
            # Get first COOP run
            coop_cond = df[df.outcome=='COOP'].iloc[0]
            cid = coop_cond.condition_id
            rep = coop_cond.replicate
            run_traj = traj[(traj.condition_id==cid) & (traj.replicate==rep)]
            if len(run_traj) > 0:
                fig, ax = plt.subplots(figsize=(10,4))
                ax.plot(run_traj.step, run_traj.mean_i, '#2A9D8F', lw=2, label='mean i-factor')
                ax.plot(run_traj.step, run_traj.cooperation_level, 'gray', lw=1.5, 
                        ls='--', label='cooperation level')
                ax.axhline(0.9, color='#2A9D8F', lw=1, ls=':', alpha=0.7)
                ax.axhline(1.0, color='black', lw=0.8, ls='--', alpha=0.4, label='i=1.0')
                ax.set_xlabel('Step'); ax.set_ylabel('i-factor')
                ax.set_title(f'Recorded COOP trajectory (condition {cid}, replicate {rep})',
                             fontweight='bold')
                ax.legend(fontsize=9)
                plt.tight_layout()
                plt.show()
            break

## Summary of Results

| Prediction | Test | Result | Status |
|-----------|------|--------|--------|
| Bimodal i distribution | Hartigan dip, p < 0.05 | dip=0.047, p≈0 | ✅ PASS |
| Threshold universality | ANOVA p > 0.05 | F=1.91, p=0.15 | ✅ PASS |
| Crossing time ∝ ε | Regression p < 0.05 | Resource dominates | ⬜ NULL |
| Strong negative correlation | r < −0.90 | r = −0.866 | ⚠️ NEAR (asymmetric distribution) |
| φ×β predicts crossing | Sens/spec > 0.75 | 1.0 / 0.997 | ✅ PASS |
| ESS attractor stability | Recovery > 85% | 34/34 (100%) | ✅ PASS |

4/5 directional predictions confirmed. The null result on Prediction 3 is theoretically informative (ecological opportunity drives transitions, not mutation rate). The near-miss on Prediction 4 reflects the asymmetric parameter grid (79% COMP conditions), not a model failure.

---

## References

- Allee, W.C. (1931). *Animal Aggregations*. University of Chicago Press.
- Axelrod, R. & Hamilton, W.D. (1981). The evolution of cooperation. *Science*, 211, 1390–1396.
- Courchamp, F., Clutton-Brock, T. & Grenfell, B. (1999). Inverse density dependence and the Allee effect. *Trends in Ecology & Evolution*, 14(10), 405–410.
- Evans, L.W. (2026). *Blueprint for Harmony*. Cooperation Press. ISBN 978-969-8992-10-1.
- Evans, L.W. (2026). MELV ABM V2.1. Zenodo. DOI: [10.5281/zenodo.19422174](https://doi.org/10.5281/zenodo.19422174)
- Gore, J., Youk, H. & van Oudenaarden, A. (2009). Snowdrift game dynamics and facultative cheating in yeast. *Nature*, 459, 253–256.
- Michod, R.E. (1999). *Darwinian Dynamics*. Princeton University Press.
- Nadell, C.D., Drescher, K. & Foster, K.R. (2016). Spatial structure, cooperation and competition in biofilms. *Nature Reviews Microbiology*, 14, 589–600.
- Nowak, M.A. & May, R.M. (1992). Evolutionary games and spatial chaos. *Nature*, 359, 826–829.
- Tarnita, C.E. et al. (2011). Multiple strategies in structured populations. *PNAS*, 108(6), 2334–2337.

---

*Laurence W. Evans | ORCID: 0009-0001-0963-1840*  
*naturesholism.substack.com | @NaturesHolism*